# EX05 — Results Analysis Notebook

Loads the raw `results/*.json`, renders the comparison table, computes the
economic break-even, and (re)generates the figures. Run from the repo root:
`uv run jupyter lab notebooks/analysis.ipynb` (or open in VS Code).

All logic comes through the package's SDK/services — the notebook only
orchestrates and displays.

In [ ]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')  # run from repo root so results/ and figures/ resolve
from airllm_bench.services.analyze import load_results, markdown_table
from airllm_bench.services.economics import Scenario, summarize
from airllm_bench.shared.config import Config

rows = load_results('results')
print(markdown_table(rows))

In [ ]:
# Economic analysis (assumptions from config/setup.json)
scn = Scenario.from_config(Config().section('economics'))
print(summarize(scn))
print('break-even (requests):', scn.break_even())

In [ ]:
# Regenerate every figure from the raw results + economic model
from airllm_bench.services.analyze import make_figures

for p in make_figures(rows, scn):
    print('wrote', p)

In [ ]:
# Display the key figures inline
from IPython.display import Image, display

for fig in ['throughput.png', 'peak_ram.png', 'ttft_vs_tpot.png', 'break_even.png', 'roofline.png']:
    path = os.path.join('figures', fig)
    if os.path.exists(path):
        display(Image(path))

## Reading the results

- **Baseline** fails (OS-killed during load) — the memory-capacity bottleneck.
- **AirLLM fp16** runs in ~3.6 GB but ~144 s/token (disk-I/O-bound).
- **Ollama Q4** fits RAM → ~4.49 tok/s (the practical local choice).
- **Ollama Q8** overflows RAM → disk-paged → ~0.03 tok/s (the RAM cliff).

Decisive factor: whether the working set fits in RAM, not the bit-width.